# Transformer T6_All_Features: All Features

One-hour-ahead EV charging demand prediction with a 24-hour lookback. Run cells from top to bottom. Results are saved in this experiment's own Google Drive folder.

## Import Libraries and Mount Google Drive

In [ ]:
# ============================================================
# TRANSFORMER FEATURE ABLATION STUDY - 2019
# T1: Historical
# T2: Historical + Traffic
# T3: Historical + Weather
# T4: Historical + EV Sales
# T5: Historical + Traffic + Weather
# T6: All Features
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, json, random, gc, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# ============================================================


## Experiment Settings and Input Features

In [ ]:
# 1. SETTINGS
# ============================================================

SEED = 42
LOOKBACK = 24
EPOCHS = 200
BATCH_SIZE = 32
LEARNING_RATE = 0.001

ROOT = "/content/drive/MyDrive/EV_Load_Forecasting_FYP"

DATA_PATH = os.path.join(
    ROOT, "Processed Data", "Hourly",
    "experiment_c_final_2019.csv"
)

OUTPUT_DIR = os.path.join(
    ROOT, "Experiments",
    "Transformer_Feature_Ablation_2019"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET = "hourly_energy_kwh"
TRAFFIC = "total_traffic"
SALES = "monthly_ev_sales_lag1"

WEATHER = [
    "clouds_lag1d",
    "dewpt_lag1d",
    "ghi_lag1d",
    "max_uv_lag1d",
    "precip_lag1d",
    "rh_lag1d",
    "slp_lag1d",
    "temp_lag1d",
    "wind_gust_spd_lag1d",
    "wind_spd_lag1d"
]

EXPERIMENTS = {"T6_All_Features": ["demand_input", "traffic_input"] + WEATHER + ["sales_input"]}

# ============================================================


## Load and Inspect the 2019 Dataset

In [ ]:
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date_hour"]
)

df = df.sort_values("date_hour").reset_index(drop=True)

required = ["date_hour", TARGET, TRAFFIC, SALES] + WEATHER

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required].copy()

for c in required[1:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

assert len(df) == 8016
assert df[TARGET].isna().sum() == 206
assert not df["date_hour"].duplicated().any()

expected_dates = pd.date_range(
    df["date_hour"].min(),
    df["date_hour"].max(),
    freq="h"
)

assert np.array_equal(
    expected_dates.to_numpy(),
    df["date_hour"].to_numpy()
), "Hourly timestamps are not continuous."

N = len(df)
TRAIN_END = int(N * 0.70)
VAL_END = int(N * 0.85)

assert TRAIN_END == 5611
assert VAL_END == 6813

print("Dataset:", df.shape)
print("Train boundary:", TRAIN_END)
print("Validation boundary:", VAL_END)

# ============================================================


## Causal Missing-Value Handling

In [ ]:
# 3. CAUSAL MISSING-VALUE HANDLING
# ============================================================

processed = df.copy()

processed["target_original"] = processed[TARGET]

processed["demand_input"] = processed[TARGET].ffill()
processed["traffic_input"] = processed[TRAFFIC].ffill()
processed["sales_input"] = processed[SALES].ffill()

processed[WEATHER] = processed[WEATHER].ffill()

external = ["traffic_input", "sales_input"] + WEATHER

for col in external:

    if processed[col].isna().any():

        train_median = (
            processed.iloc[:TRAIN_END][col].median()
        )

        if pd.isna(train_median):
            raise ValueError(
                f"No training values for {col}"
            )

        processed[col] = (
            processed[col].fillna(train_median)
        )

if processed["demand_input"].isna().any():
    raise ValueError(
        "Leading historical demand values are missing."
    )

assert processed[
    ["demand_input"] + external
].notna().all().all()

processed.to_csv(
    os.path.join(OUTPUT_DIR, "processed_dataset.csv"),
    index=False
)

# ============================================================


## Define Transformer Architecture

In [ ]:
# 4. TRANSFORMER ARCHITECTURE
# ============================================================

def build_transformer(
    lookback,
    n_features,
    d_model=32,
    num_heads=2,
    ff_dim=64,
    dropout=0.1
):

    inputs = layers.Input(
        shape=(lookback, n_features)
    )

    # Project features into transformer dimension
    x = layers.Dense(d_model)(inputs)

    # Learnable positional embeddings
    positions = tf.range(
        start=0,
        limit=lookback,
        delta=1
    )

    position_embedding = layers.Embedding(
        input_dim=lookback,
        output_dim=d_model
    )(positions)

    x = x + position_embedding

    # Self-attention block
    attention = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout
    )(x, x)

    attention = layers.Dropout(dropout)(attention)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x + attention)

    # Feed-forward block
    ff = layers.Dense(
        ff_dim,
        activation="relu"
    )(x)

    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x + ff)

    # Aggregate sequence representation
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(1)(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
            clipnorm=1.0
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model


# ============================================================


## Create 24-Hour Input Sequences

In [ ]:
# 5. SEQUENCE GENERATION
# ============================================================

def create_sequences(X, y, timestamps):

    result = {
        name: {"X": [], "y": [], "time": []}
        for name in ["train", "val", "test"]
    }

    for i in range(LOOKBACK, len(y)):

        if not np.isfinite(y[i]):
            continue

        window = X[i-LOOKBACK:i]

        if not np.isfinite(window).all():
            continue

        if i < TRAIN_END:
            split = "train"
        elif i < VAL_END:
            split = "val"
        else:
            split = "test"

        result[split]["X"].append(window)
        result[split]["y"].append(y[i])
        result[split]["time"].append(timestamps[i])

    for split in result:

        result[split]["X"] = np.asarray(
            result[split]["X"],
            dtype=np.float32
        ).reshape(-1, LOOKBACK, X.shape[1])

        result[split]["y"] = np.asarray(
            result[split]["y"],
            dtype=np.float32
        ).reshape(-1, 1)

        result[split]["time"] = pd.DatetimeIndex(
            result[split]["time"]
        )

    return result


# ============================================================


## Define Evaluation Metrics

In [ ]:
# 6. EVALUATION
# ============================================================

def evaluate(model, X, y, timestamps, scaler_y):

    pred_scaled = model.predict(X, verbose=0)

    actual = scaler_y.inverse_transform(
        y.reshape(-1, 1)
    ).ravel()

    predicted = scaler_y.inverse_transform(
        pred_scaled.reshape(-1, 1)
    ).ravel()

    metrics = {
        "Samples": len(actual),
        "RMSE (kWh)": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "MAE (kWh)": mean_absolute_error(
            actual, predicted
        ),
        "R² Score": r2_score(actual, predicted)
    }

    predictions = pd.DataFrame({
        "date_hour": timestamps,
        "actual_kwh": actual,
        "predicted_kwh": predicted
    })

    return metrics, predictions


# ============================================================


## Select This Experiment

In [ ]:
exp_name = "T6_All_Features"
features = EXPERIMENTS[exp_name]
all_results = []
all_predictions = {}


## Initialise Experiment and Random Seed

In [ ]:
print("\n" + "=" * 75)
print("TRAINING:", exp_name)
print("Features:", features)
print("=" * 75)

tf.keras.backend.clear_session()
gc.collect()

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

exp_dir = os.path.join(OUTPUT_DIR, exp_name)
os.makedirs(exp_dir, exist_ok=True)


## Scale Inputs and Target Using Training Data Only

In [ ]:
# Fit scalers on training data only
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_X.fit(
    processed.iloc[:TRAIN_END][features]
)

scaler_y.fit(
    processed.iloc[:TRAIN_END][["target_original"]]
    .dropna()
)

X_scaled = scaler_X.transform(
    processed[features]
).astype(np.float32)

y_scaled = np.full(
    N, np.nan, dtype=np.float32
)

valid = processed["target_original"].notna().to_numpy()

y_scaled[valid] = scaler_y.transform(
    processed.loc[valid, ["target_original"]]
).ravel().astype(np.float32)


## Generate 24-Hour Sequences and Verify Shapes

In [ ]:
# Generate identical target timestamps
data = create_sequences(
    X_scaled,
    y_scaled,
    processed["date_hour"].to_numpy()
)

X_train = data["train"]["X"]
y_train = data["train"]["y"]

X_val = data["val"]["X"]
y_val = data["val"]["y"]

X_test = data["test"]["X"]
y_test = data["test"]["y"]

print("\nSequence shapes:")
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

assert X_train.shape == (
    5403, LOOKBACK, len(features)
)

assert X_val.shape == (
    1187, LOOKBACK, len(features)
)

assert X_test.shape == (
    1196, LOOKBACK, len(features)
)


## Build Model and Set Training Callbacks

In [ ]:
# Build model
model = build_transformer(
    LOOKBACK,
    len(features)
)

model_path = os.path.join(
    exp_dir, "best_transformer.keras"
)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        model_path,
        monitor="val_loss",
        save_best_only=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=10,
        min_lr=1e-6
    )
]


## Train Transformer

In [ ]:
# Train
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=1
)


## Export Training History

In [ ]:
# Save training history
pd.DataFrame(history.history).to_csv(
    os.path.join(exp_dir, "training_history.csv"),
    index=False
)


## Evaluate Training, Validation and Test Sets

In [ ]:
# Use restored best validation weights
# (equivalent to saved best checkpoint)
split_results = []

for split_name in ["train", "val", "test"]:

    metrics, predictions = evaluate(
        model,
        data[split_name]["X"],
        data[split_name]["y"],
        data[split_name]["time"],
        scaler_y
    )

    metrics["Dataset"] = split_name
    split_results.append(metrics)

    predictions.to_csv(
        os.path.join(
            exp_dir,
            f"{split_name}_predictions.csv"
        ),
        index=False
    )

    if split_name == "test":

        all_predictions[exp_name] = predictions

        all_results.append({
            "Model": exp_name,
            **metrics
        })

pd.DataFrame(split_results).to_csv(
    os.path.join(exp_dir, "all_split_metrics.csv"),
    index=False
)


## Save Input and Target Scalers

In [ ]:
# Save scalers
joblib.dump(
    scaler_X,
    os.path.join(exp_dir, "scaler_X.joblib")
)

joblib.dump(
    scaler_y,
    os.path.join(exp_dir, "scaler_y.joblib")
)


## Plot Training and Validation Loss

In [ ]:
# Training plot
plt.figure(figsize=(10, 5))
plt.plot(
    history.history["loss"],
    label="Training"
)
plt.plot(
    history.history["val_loss"],
    label="Validation"
)
plt.title(f"{exp_name} - Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Scaled MSE")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(exp_dir, "training_loss.png"),
    dpi=300
)
plt.show()
plt.close()


## Plot Actual Versus Predicted Test Demand

In [ ]:
# Test prediction plot
test_pred = all_predictions[exp_name]

plt.figure(figsize=(14, 5))
plt.plot(
    test_pred["date_hour"],
    test_pred["actual_kwh"],
    label="Actual",
    linewidth=1
)
plt.plot(
    test_pred["date_hour"],
    test_pred["predicted_kwh"],
    label="Predicted",
    linewidth=1
)
plt.title(f"{exp_name} - Test Predictions")
plt.xlabel("Date")
plt.ylabel("Charging Demand (kWh)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        exp_dir, "test_actual_vs_predicted.png"
    ),
    dpi=300
)
plt.show()
plt.close()


## Save Experiment Manifest

In [ ]:
manifest = {
    "experiment": exp_name,
    "features": features,
    "n_features": len(features),
    "lookback": LOOKBACK,
    "horizon": 1,
    "split": [0.70, 0.15, 0.15],
    "architecture": "Transformer Encoder",
    "d_model": 32,
    "heads": 2,
    "ff_dim": 64,
    "dropout": 0.1,
    "loss": "mse",
    "epochs_max": EPOCHS,
    "best_epoch": int(
        np.argmin(history.history["val_loss"]) + 1
    ),
    "seed": SEED,
    "model_parameters": int(model.count_params())
}

with open(
    os.path.join(exp_dir, "manifest.json"),
    "w"
) as f:
    json.dump(manifest, f, indent=4)

print("\nCompleted:", exp_name)


## Experiment Complete

The trained model, metrics, predictions, plots, scalers and manifest are saved in Google Drive. Use the separate comparison notebook after all six experiments have completed.